# 02 · ASTRA 追踪 (工作区 / 设置 / 运行)

本 notebook 合并了原 00/02/03: 环境自检 -> 输入卡 (表单或文本) ->
运行 ASTRA -> 输出清单。与官方 astra 程序对应。

In [ ]:
%run _bootstrap.py

In [ ]:
# 环境自检
import numpy, scipy, matplotlib, pandas
print("numpy", numpy.__version__, "| scipy", scipy.__version__,
      "| matplotlib", matplotlib.__version__, "| pandas", pandas.__version__)
print()
print("ASTRA    :", ASTRA_EXE)
print("Generator:", GENERATOR_EXE)
print("工作目录 :", SIM_DIR)
print("下一步: 打开 01_generator.ipynb 生成初始束团。")

## 输入卡 — 表单模式

按 namelist 分组, 覆盖手册第 6 章全部 13 个 namelist; 基础组已填入
可运行的默认值。

In [ ]:
from astra_tools.widgets.forms import namelist_form
from astra_tools.deck.metadata import summary
print(summary())
print()
print('用法: namelist_form("NEWRUN", only=[...])  # 常用参数')
print('      namelist_form("NEWRUN")              # 全部参数')

In [ ]:
# 常用追踪参数表单 (逐组生成)
forms, getters = {}, {}
# 基础组: 以可运行的默认值作种子 (纯漂移, 产生全部输出文件)
_basic_seed = {
    "NEWRUN": {"Track_All": True, "Auto_Phase": True,
               "check_ref_part": False, "H_max": 0.001, "H_min": 0.0,
               "Xoff": 0.0, "Yoff": 0.0},
    "OUTPUT": {"ZSTART": 0.0, "ZSTOP": 1.5, "Zemit": 100, "Zphase": 1,
               "RefS": True, "EmitS": True, "PhaseS": True, "SigmaS": True},
}
groups = {
    "NEWRUN": ["Head", "RUN", "Distribution", "Track_All", "Auto_Phase",
               "check_ref_part", "H_max", "H_min", "Xoff", "Yoff", "Toff"],
    "OUTPUT": ["ZSTART", "ZSTOP", "Zemit", "Zphase", "RefS", "EmitS", "PhaseS",
               "SigmaS", "C_EmitS", "Lsub_cor", "Binary", "High_res"],
    "CHARGE": ["LSPCH", "Nrad", "Cell_var", "Nlong_in", "min_grid", "Max_Scale"],
    "CAVITY": ["LEfield", "File_Efield", "C_pos", "Nue", "MaxE", "Phi"],
    "SOLENOID": ["LBfield", "File_Bfield", "S_pos", "MaxB", "S_higher_order"],
    "WAKE": ["LWAKE", "File_Wakefield", "W_pos"],
    "APERTURE": ["LAPERT", "File_APERTURE", "AP_radius"],
}
for gname, only in groups.items():
    wmap, getter = namelist_form(gname, values=_basic_seed.get(gname, {}), only=only)
    forms[gname] = wmap
    getters[gname] = getter
print("各组表单已生成 (基础组已填入可运行默认值)。")

In [ ]:
# 写 astra.in (表单模式)
from astra_tools.namelist.write import write_input_deck

blocks = {}
for gname, getter in getters.items():
    # 基础组 (NEWRUN/OUTPUT) 全量写入; 其余组只写用户改动过的参数
    vals = getter(changed_only=(gname not in _basic_seed))
    if vals:
        blocks[gname] = vals
blocks.setdefault("NEWRUN", {})["Distribution"] = "'bunch.ini'"  # 相对路径
write_input_deck(blocks, SIM_DIR / "astra.in",
                 header="generated by astra-notebook (02_astra)")
print("astra.in 已写入, 包含 namelist:", list(blocks))
print()
print((SIM_DIR / "astra.in").read_text())

**文本模式**: 把现成 .in 文件复制为 `data/workspace/astra.in` 即可,
跳过表单直接运行下方单元。

## 运行 ASTRA

运行 data/workspace/astra.in, 实时日志流式输出, 结束后列出输出文件。

In [ ]:
from astra_tools.run import run_program, discover_outputs

input_path = SIM_DIR / "astra.in"
if not input_path.exists():
    raise FileNotFoundError("astra.in 不存在 — 请先运行上方设置单元")
run_program(ASTRA_EXE, SIM_DIR, input_file="astra.in")
print("ASTRA 运行完成。")

In [ ]:
import pandas as pd
outs = discover_outputs(SIM_DIR, "astra", run="001")
rows = []
for key, val in outs.items():
    if isinstance(val, list):
        rows += [(key, str(f.name), f.stat().st_size) for f in val]
    elif val is not None:
        rows += [(key, str(val.name), val.stat().st_size)]
pd.DataFrame(rows, columns=["类型", "文件", "大小(字节)"])

下一步: `03_postpro.ipynb` (相空间与统计) / `04_lineplot.ipynb` (演化)。